# Cyclospora outbreak

## CDC Surveillance data

In [1]:
import requests
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
import pandas as pd
import re

In [2]:
playwright = await async_playwright().start()
browser = await playwright.chromium.launch(headless=True)

In [3]:
page = await browser.new_page()

In [4]:
await page.goto("https://www.cdc.gov/cyclosporiasis/php/surveillance/index.html")

<Response url='https://www.cdc.gov/cyclosporiasis/php/surveillance/index.html' request=<Request url='https://www.cdc.gov/cyclosporiasis/php/surveillance/index.html' method='GET'>>

In [5]:
async with page.expect_download() as download_info:
    await page.click('a[aria-label="Download this data in a CSV file format."]')

download = await download_info.value
await download.save_as("data/data.csv")
print("Downloaded to:", await download.path())

Downloaded to: /tmp/playwright-artifacts-uSR2xo/a962ac34-8f1a-4195-8632-7501bfffb2c6


In [6]:
df = pd.read_csv("data/data.csv")

In [7]:
df.head()

,Location,Number of Sick People
0,Alaska,1 to 10
1,Arizona,1 to 10
2,Arkansas,1 to 10
3,California,1 to 10
4,Colorado,1 to 10


In [8]:
df['min'] = df['Number of Sick People'].str.extract(r'(\d+)\s+to\s+\d+').astype(int)
df['max'] = df['Number of Sick People'].str.extract(r'\d+\s+to\s+(\d+)').astype(int)

In [9]:
df['average'] = df[['min', 'max']].mean(axis=1)

In [10]:
df.sort_values(by='average', ascending=False, inplace=True)
df.head()

,Location,Number of Sick People,min,max,average
16,Michigan,501 to 900,501,900,700.5
21,New York,161 to 300,161,300,230.5
22,North Carolina,81 to 160,81,160,120.5
20,New Jersey,31 to 80,31,80,55.5
28,Texas,31 to 80,31,80,55.5


In [11]:
df.to_csv("data/clean_data.csv", index=False)

### Cyclospora confirmed cases and notes

In [12]:
await page.goto("https://www.cdc.gov/cyclosporiasis/php/surveillance/index.html")

<Response url='https://www.cdc.gov/cyclosporiasis/php/surveillance/index.html' request=<Request url='https://www.cdc.gov/cyclosporiasis/php/surveillance/index.html' method='GET'>>

In [13]:
fast_facts = await page.locator('div.dfe-section[data-section="cdc_generic_section_3"]').inner_text()
print(fast_facts)


2026 fast facts

As of July 13, 2026:

U.S. cases reported to CDC: 1,645
Hospitalizations: 141
Deaths: 0
States reporting cases: 34


In [14]:
fast_facts

'2026 fast facts\n\nAs of July 13, 2026:\n\nU.S. cases reported to CDC: 1,645\nHospitalizations: 141\nDeaths: 0\nStates reporting cases: 34'

In [15]:
labels = ['last_updated', 'total_cases', 'hospitalizations', 'deaths', 'states_reporting']

as_of_text = await page.locator('div.dfe-section[data-section="cdc_generic_section_3"] p').inner_text()

as_of_date = re.search(r'As of (.+?):', as_of_text).group(1).strip()

items = await page.locator('div.dfe-section[data-section="cdc_generic_section_3"] ul.nested-list li').all()

values = [as_of_date] + [((await item.inner_text()).split(':')[1].strip()) for item in items]

cdc_data = dict(zip(labels, values))
print(cdc_data)

{'last_updated': 'July 13, 2026', 'total_cases': '1,645', 'hospitalizations': '141', 'deaths': '0', 'states_reporting': '34'}


In [16]:
df_cdc = pd.DataFrame([cdc_data])

In [17]:
df_cdc.to_csv("data/cdc_notes.csv", index=False)

### Add last updated to CDC data

In [18]:
df['Last Updated'] = df_cdc['last_updated'][0]

In [19]:
df

,Location,Number of Sick People,min,max,average,Last Updated
16,Michigan,501 to 900,501,900,700.5,"July 13, 2026"
21,New York,161 to 300,161,300,230.5,"July 13, 2026"
22,North Carolina,81 to 160,81,160,120.5,"July 13, 2026"
20,New Jersey,31 to 80,31,80,55.5,"July 13, 2026"
28,Texas,31 to 80,31,80,55.5,"July 13, 2026"
8,Illinois,31 to 80,31,80,55.5,"July 13, 2026"
12,Kentucky,31 to 80,31,80,55.5,"July 13, 2026"
9,Indiana,31 to 80,31,80,55.5,"July 13, 2026"
30,Virginia,11 to 30,11,30,20.5,"July 13, 2026"
27,Tennessee,11 to 30,11,30,20.5,"July 13, 2026"


# Local CDC Data

## Michigan

### First, drop duplicate data from CDC

In [20]:
states_to_drop = ['Michigan', 'Alabama', 'Ohio', 'Louisiana','Indiana', 'Maryland', 'Mississippi']

df = df.drop(df[df['Location'].isin(states_to_drop)].index)

In [21]:
df.head()

,Location,Number of Sick People,min,max,average,Last Updated
21,New York,161 to 300,161,300,230.5,"July 13, 2026"
22,North Carolina,81 to 160,81,160,120.5,"July 13, 2026"
20,New Jersey,31 to 80,31,80,55.5,"July 13, 2026"
28,Texas,31 to 80,31,80,55.5,"July 13, 2026"
8,Illinois,31 to 80,31,80,55.5,"July 13, 2026"


In [22]:
await page.goto("https://www.michigan.gov/mdhhs/keep-mi-healthy/infectious-diseases/infectious-disease-outbreaks")

<Response url='https://www.michigan.gov/mdhhs/keep-mi-healthy/infectious-diseases/infectious-disease-outbreaks' request=<Request url='https://www.michigan.gov/mdhhs/keep-mi-healthy/infectious-diseases/infectious-disease-outbreaks' method='GET'>>

In [23]:
total_cases = await page.inner_text('p:has(strong:text("Total Cases:"))')
print(total_cases)


Total Cases: 6,148
As of July 16, 2026, 102 reported cases indicated they had been hospitalized.


In [24]:
match = re.search(r'Total Cases:\s*([\d,]+)', total_cases)
cases = match.group(1) 

michigan_cases = int(cases.replace(',', '')) 

print(cases)     
print(michigan_cases) 

6,148
6148


In [25]:
last_updated = await page.inner_text('p:has(strong:text("Last updated:"))')
date_match = re.search(r'(\w+ \d+, \d{4})', last_updated)
last_updated_Michigan = date_match.group(1) if date_match else None
print(last_updated_Michigan)

July 20, 2026


## Add local Michigan data

In [26]:
df.head()

,Location,Number of Sick People,min,max,average,Last Updated
21,New York,161 to 300,161,300,230.5,"July 13, 2026"
22,North Carolina,81 to 160,81,160,120.5,"July 13, 2026"
20,New Jersey,31 to 80,31,80,55.5,"July 13, 2026"
28,Texas,31 to 80,31,80,55.5,"July 13, 2026"
8,Illinois,31 to 80,31,80,55.5,"July 13, 2026"


In [27]:
new_row = {
    'Location': 'Michigan',
    'Number of Sick People': michigan_cases,
    'min': michigan_cases,
    'max': michigan_cases,
    'average': michigan_cases,
    'Last Updated': last_updated_Michigan 
}

df_local = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

In [28]:
df_local

,Location,Number of Sick People,min,max,average,Last Updated
0,New York,161 to 300,161,300,230.5,"July 13, 2026"
1,North Carolina,81 to 160,81,160,120.5,"July 13, 2026"
2,New Jersey,31 to 80,31,80,55.5,"July 13, 2026"
3,Texas,31 to 80,31,80,55.5,"July 13, 2026"
4,Illinois,31 to 80,31,80,55.5,"July 13, 2026"
5,Kentucky,31 to 80,31,80,55.5,"July 13, 2026"
6,Virginia,11 to 30,11,30,20.5,"July 13, 2026"
7,Tennessee,11 to 30,11,30,20.5,"July 13, 2026"
8,Oklahoma,11 to 30,11,30,20.5,"July 13, 2026"
9,Kansas,11 to 30,11,30,20.5,"July 13, 2026"


## Alabama

In [29]:
# await page.goto("https://www.alabamapublichealth.gov/infectiousdiseases/cases.html")

In [30]:
# await page.wait_for_load_state("networkidle", timeout=30000)

### Manually adding Alabama data

In [31]:
# df_local.head()

In [32]:
# alabama = pd.DataFrame([{"Location": "Alabama", "Number of Sick People": 2, "min": 2, "max": 2, "average": 2, "Last Updated": "July 15, 2026"}])

In [33]:
# df_local = pd.concat([df_local, alabama], ignore_index=True)

In [34]:
# df_local

In [35]:
# df_local.to_csv("data/clean_data_local.csv", index=False)

## Add local stations data on the Sheet

In [36]:
sheet_id = "14INvC79GAOzaEdB_9mRztD7Mw_PDLlYYMwVVAB3e_-k"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

In [37]:
stations = pd.read_csv(url)

In [38]:
stations

,Location,Number of Sick People,min,max,average,Last Updated
0,Ohio,1274,1274,1274,1274,"July 18, 2026"
1,Louisiana,55,55,55,55,"July 16, 2026"
2,Alabama,2,2,2,2,"July 18, 2026"
3,Indiana,366,366,366,366,"July 17, 2026"
4,Maryland,69,69,69,69,"July 14, 2026"
5,Florida,96,96,96,96,"July 11, 2026"
6,Mississippi,5,5,5,5,"July 16, 2026"


In [39]:
df_local = pd.concat([df_local, stations], ignore_index=True)

In [40]:
df_local

,Location,Number of Sick People,min,max,average,Last Updated
0,New York,161 to 300,161,300,230.5,"July 13, 2026"
1,North Carolina,81 to 160,81,160,120.5,"July 13, 2026"
2,New Jersey,31 to 80,31,80,55.5,"July 13, 2026"
3,Texas,31 to 80,31,80,55.5,"July 13, 2026"
4,Illinois,31 to 80,31,80,55.5,"July 13, 2026"
5,Kentucky,31 to 80,31,80,55.5,"July 13, 2026"
6,Virginia,11 to 30,11,30,20.5,"July 13, 2026"
7,Tennessee,11 to 30,11,30,20.5,"July 13, 2026"
8,Oklahoma,11 to 30,11,30,20.5,"July 13, 2026"
9,Kansas,11 to 30,11,30,20.5,"July 13, 2026"


In [41]:
df_local.to_csv("data/clean_data_local.csv", index=False)